# On Calibration of Modern Neural Networks — Reproduction

**Paper:** Guo, Pleiss, Sun, Weinberger. *On Calibration of Modern Neural Networks.* ICML 2017. [arXiv:1706.04599](https://arxiv.org/abs/1706.04599)

This notebook reproduces the **core empirical finding** of the paper on a small scale:

> Modern, high-capacity neural networks tend to be *overconfident* — their predicted confidence no longer matches their actual accuracy — even though they are more accurate than older, shallower networks. A single learned scalar, **temperature scaling**, fixes most of this miscalibration without changing accuracy at all.

### What we reproduce
- **Figure 1** — confidence histograms + reliability diagrams contrasting a shallow, well-calibrated network with a deep, miscalibrated one (here: a 5-layer ConvNet vs. a 56-layer ResNet on CIFAR-100).
- **Section 2** — Expected Calibration Error (ECE, Eq. 3), Maximum Calibration Error (MCE, Eq. 5), and NLL (Eq. 6).
- **Section 4.2 / Eq. 9** — Temperature scaling, fit by minimizing NLL on a held-out validation set.
- **Section 4.1 / Eq. 7** — Histogram binning, as a second calibration method for comparison.
- **Table 1 / Figure 4** — a small-scale version of the before/after calibration comparison table and reliability diagrams.
- *(Optional, at the end)* — a mini version of **Figure 2** (ECE vs. network depth).

### What is scaled down vs. the paper (and why)
| Paper | This notebook | Why |
|---|---|---|
| CIFAR-100, CIFAR-10, SVHN, ImageNet, Birds, Cars, 4 NLP datasets | CIFAR-100 only | keep runtime to well under an hour on a free Colab GPU |
| ResNet-110, Wide ResNet-32, DenseNet-40, LeNet-5, etc. (multiple architectures per dataset) | one shallow ConvNet + one 56-layer ResNet | still gives the shallow-vs-deep contrast the paper's Figure 1 is built on |
| ~200–500 training epochs with careful LR schedules | ~20–40 epochs | good enough to reach the *qualitative* miscalibration regime; final accuracy will be lower than the paper's |
| Full one-vs-rest multiclass histogram binning, isotonic regression, BBQ, matrix/vector scaling | temperature scaling (full, exact) + a simplified top-label histogram binning | keeps the notebook focused; temperature scaling is the paper's headline result anyway |

The qualitative conclusions (deeper net ⇒ higher ECE despite higher accuracy; temperature scaling ⇒ large ECE drop, zero accuracy change) should still hold at this scale, but exact numbers will differ from Table 1.

### Runtime
Designed for **Google Colab with a GPU runtime** (`Runtime → Change runtime type → GPU`). Total runtime: roughly 15–25 minutes.


## 1. Setup

In [ ]:
# Uncomment if running somewhere torch/torchvision aren't already installed (Colab has them preinstalled).
# !pip install -q torch torchvision matplotlib pandas numpy


In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using device:", device)
if device.type == "cpu":
    print("WARNING: no GPU detected. On Colab, go to Runtime > Change runtime type > GPU.")


## 2. Data: CIFAR-100

The paper uses a **45,000 / 5,000 / 10,000** train/validation/test split for CIFAR-100 (Section 5) — the validation
set is used later to *fit* the calibration methods, never to train the network itself. We reproduce that exact split.


In [ ]:
CIFAR100_MEAN = (0.5071, 0.4865, 0.4409)
CIFAR100_STD = (0.2673, 0.2564, 0.2762)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

full_train_aug = torchvision.datasets.CIFAR100(root="./data", train=True, download=True, transform=train_transform)
full_train_plain = torchvision.datasets.CIFAR100(root="./data", train=True, download=True, transform=eval_transform)
test_set = torchvision.datasets.CIFAR100(root="./data", train=False, download=True, transform=eval_transform)

# Fixed 45,000 / 5,000 split, matching the paper's CIFAR-100 protocol (Section 5).
split_generator = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(full_train_aug), generator=split_generator).tolist()
train_idx, val_idx = perm[:45000], perm[45000:]

train_set = Subset(full_train_aug, train_idx)
val_set = Subset(full_train_plain, val_idx)  # validation set must NOT use train-time augmentation

BATCH_SIZE = 128
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=(device.type == "cuda"))
val_loader = DataLoader(val_set, batch_size=256, shuffle=False, num_workers=2, pin_memory=(device.type == "cuda"))
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2, pin_memory=(device.type == "cuda"))

NUM_CLASSES = 100
print(f"train={len(train_set)}  val={len(val_set)}  test={len(test_set)}")


## 3. Models

The paper's Figure 1 contrasts a **5-layer LeNet** (well-calibrated) with a **110-layer ResNet** (miscalibrated) on
CIFAR-100. We reproduce the same *contrast in capacity*, using a smaller ResNet (56 layers, still `6n+2` as in
He et al. 2016) to keep training time reasonable, while keeping the shallow LeNet-style network as the low-capacity
baseline.


In [ ]:
class SmallConvNet(nn.Module):
    """5-layer LeNet-style ConvNet (2 conv + 3 fc layers), adapted for 32x32x3 CIFAR input."""

    def __init__(self, num_classes=100):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, kernel_size=5)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes * self.expansion, 1, stride, bias=False),
                nn.BatchNorm2d(planes * self.expansion),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out)


class CifarResNet(nn.Module):
    """Standard CIFAR ResNet (He et al. 2016), depth = 6n + 2, e.g. depth=56 -> n=9."""

    def __init__(self, depth=56, num_classes=100):
        super().__init__()
        assert (depth - 2) % 6 == 0, "depth must be of the form 6n+2"
        n = (depth - 2) // 6
        self.in_planes = 16
        self.conv1 = nn.Conv2d(3, 16, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.layer1 = self._make_layer(16, n, stride=1)
        self.layer2 = self._make_layer(32, n, stride=2)
        self.layer3 = self._make_layer(64, n, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, num_classes)

    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes * BasicBlock.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        return self.fc(out)


## 4. Training utilities

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        n += xb.size(0)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        total_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        n += xb.size(0)
    return total_loss / n, correct / n


def fit(model, train_loader, val_loader, device, epochs, lr, weight_decay, milestones):
    model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                                 weight_decay=weight_decay, nesterov=True)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=0.1)
    history = []
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, device)
        va_loss, va_acc = evaluate(model, val_loader, device)
        scheduler.step()
        history.append((epoch, tr_loss, tr_acc, va_loss, va_acc))
        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            print(f"epoch {epoch:3d}/{epochs}  train_loss={tr_loss:.3f} train_acc={tr_acc:.3f}  "
                  f"val_loss={va_loss:.3f} val_acc={va_acc:.3f}  lr={scheduler.get_last_lr()[0]:.4f}")
    return history


@torch.no_grad()
def get_logits_labels(model, loader, device):
    model.eval()
    all_logits, all_labels = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        all_logits.append(model(xb).cpu())
        all_labels.append(yb)
    return torch.cat(all_logits), torch.cat(all_labels)


## 5. Train the shallow ConvNet (LeNet-style)

Per the paper's own finding, this low-capacity network should end up reasonably well-calibrated (small ECE),
even though its accuracy is much lower than the ResNet's.


In [ ]:
CONVNET_EPOCHS = 20

convnet = SmallConvNet(NUM_CLASSES)
convnet_history = fit(
    convnet, train_loader, val_loader, device,
    epochs=CONVNET_EPOCHS, lr=0.01, weight_decay=5e-4,
    milestones=[int(CONVNET_EPOCHS * 0.5), int(CONVNET_EPOCHS * 0.75)],
)
test_loss, test_acc = evaluate(convnet, test_loader, device)
print(f"SmallConvNet test accuracy: {test_acc:.4f}")


## 6. Train the deep ResNet

This is the "modern" high-capacity network. Per the paper, it should reach much higher accuracy than the
ConvNet, but with *worse* calibration (higher ECE) despite that.


In [ ]:
RESNET_EPOCHS = 40

resnet = CifarResNet(depth=56, num_classes=NUM_CLASSES)
resnet_history = fit(
    resnet, train_loader, val_loader, device,
    epochs=RESNET_EPOCHS, lr=0.1, weight_decay=5e-4,
    milestones=[int(RESNET_EPOCHS * 0.5), int(RESNET_EPOCHS * 0.75)],
)
test_loss, test_acc = evaluate(resnet, test_loader, device)
print(f"CifarResNet-56 test accuracy: {test_acc:.4f}")


## 7. Calibration metrics

Straight from the paper's Section 2:

- **Reliability diagram**: partition predictions into `M` equal-width confidence bins; plot `acc(B_m)` vs. `conf(B_m)` per bin (Eq. 3's ingredients). A perfectly calibrated model traces the diagonal.
- **Expected Calibration Error (ECE)**, Eq. 3: `ECE = sum_m (|B_m| / n) * |acc(B_m) - conf(B_m)|`.
- **Maximum Calibration Error (MCE)**, Eq. 5: `MCE = max_m |acc(B_m) - conf(B_m)|`.
- **NLL**, Eq. 6: the standard cross-entropy loss, `-sum_i log( pi_hat(y_i | x_i) )`.


In [ ]:
def bin_stats(confidences, correct, n_bins=15):
    """Returns per-bin (confidence, accuracy, proportion) for a reliability diagram."""
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    confs, accs, props = [], [], []
    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i + 1]
        if i == 0:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences > lo) & (confidences <= hi)
        prop = mask.mean()
        confs.append(confidences[mask].mean() if mask.sum() > 0 else (lo + hi) / 2)
        accs.append(correct[mask].mean() if mask.sum() > 0 else 0.0)
        props.append(prop)
    return bin_boundaries, np.array(confs), np.array(accs), np.array(props)


def compute_ece(confidences, correct, n_bins=15):
    _, confs, accs, props = bin_stats(confidences, correct, n_bins)
    return float(np.sum(props * np.abs(accs - confs)))


def compute_mce(confidences, correct, n_bins=15):
    _, confs, accs, props = bin_stats(confidences, correct, n_bins)
    gaps = np.abs(accs - confs)[props > 0]
    return float(gaps.max()) if len(gaps) else 0.0


def confidence_and_correctness(logits, labels):
    probs = F.softmax(logits, dim=1)
    confidence, prediction = probs.max(dim=1)
    correct = prediction.eq(labels).numpy().astype(np.float64)
    return confidence.numpy(), correct, prediction


def reliability_diagram(ax, confidences, correct, n_bins=15, title=""):
    bin_boundaries, confs, accs, props = bin_stats(confidences, correct, n_bins)
    bin_centers = (bin_boundaries[:-1] + bin_boundaries[1:]) / 2
    width = 1.0 / n_bins
    ax.bar(bin_centers, accs, width=width, edgecolor="black", color="#3B7DD8", label="Outputs")
    gap = confs - accs
    ax.bar(bin_centers, gap, bottom=accs, width=width, edgecolor="red", hatch="//",
           color="red", alpha=0.3, label="Gap")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
    ece = compute_ece(confidences, correct, n_bins)
    ax.text(0.05, 0.85, f"ECE={ece*100:.2f}", transform=ax.transAxes,
            bbox=dict(facecolor="white", alpha=0.8))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy")
    ax.set_title(title)
    ax.legend(loc="upper left", fontsize=8)


def confidence_histogram(ax, confidences, accuracy, title=""):
    ax.hist(confidences, bins=15, range=(0, 1), color="#3B7DD8", edgecolor="black")
    ax.axvline(confidences.mean(), color="black", linestyle="--", label="Avg. confidence")
    ax.axvline(accuracy, color="gray", linestyle="-", label="Accuracy")
    ax.set_xlim(0, 1)
    ax.set_xlabel("Confidence"); ax.set_ylabel("# samples")
    ax.set_title(title)
    ax.legend(fontsize=8)


## 8. Reproducing Figure 1: miscalibration of modern networks

Confidence histograms (top) and reliability diagrams (bottom) for the shallow ConvNet (left) vs. the deep
ResNet (right) on the CIFAR-100 **test set** — mirroring the paper's LeNet vs. 110-layer ResNet comparison.


In [ ]:
convnet_test_logits, test_labels = get_logits_labels(convnet, test_loader, device)
resnet_test_logits, _ = get_logits_labels(resnet, test_loader, device)

convnet_conf, convnet_correct, _ = confidence_and_correctness(convnet_test_logits, test_labels)
resnet_conf, resnet_correct, _ = confidence_and_correctness(resnet_test_logits, test_labels)

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
confidence_histogram(axes[0, 0], convnet_conf, convnet_correct.mean(), title="SmallConvNet (shallow) — CIFAR-100")
confidence_histogram(axes[0, 1], resnet_conf, resnet_correct.mean(), title="ResNet-56 (deep) — CIFAR-100")
reliability_diagram(axes[1, 0], convnet_conf, convnet_correct, title=f"acc={convnet_correct.mean():.3f}")
reliability_diagram(axes[1, 1], resnet_conf, resnet_correct, title=f"acc={resnet_correct.mean():.3f}")
plt.tight_layout()
plt.show()

print(f"SmallConvNet: acc={convnet_correct.mean():.4f}  ECE={compute_ece(convnet_conf, convnet_correct)*100:.2f}%  "
      f"MCE={compute_mce(convnet_conf, convnet_correct)*100:.2f}%  "
      f"NLL={F.cross_entropy(convnet_test_logits, test_labels).item():.4f}")
print(f"ResNet-56   : acc={resnet_correct.mean():.4f}  ECE={compute_ece(resnet_conf, resnet_correct)*100:.2f}%  "
      f"MCE={compute_mce(resnet_conf, resnet_correct)*100:.2f}%  "
      f"NLL={F.cross_entropy(resnet_test_logits, test_labels).item():.4f}")


If this reproduces the paper's finding, the ResNet should have **higher accuracy but a larger ECE** than the
ConvNet — its confidence histogram should be skewed hard towards 1.0, while its reliability diagram bars should
sit visibly below the diagonal (overconfidence), unlike the ConvNet's.


## 9. Temperature scaling (Eq. 9)

Given logits `z_i`, temperature scaling produces calibrated probabilities `softmax(z_i / T)` for a single
scalar `T > 0`, fit by minimizing NLL **on the validation set** (never on test data, and never touching the
network's own weights). Because `T` doesn't change the arg-max of the logits, **accuracy is provably unchanged**.


In [ ]:
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        # optimize log(T) so T = exp(log_T) stays positive by construction
        self.log_T = nn.Parameter(torch.zeros(1))

    @property
    def T(self):
        return torch.exp(self.log_T)

    def forward(self, logits):
        return logits / self.T

    def fit(self, logits, labels, max_iter=100, lr=0.01):
        logits, labels = logits.detach(), labels.detach()
        optimizer = torch.optim.LBFGS([self.log_T], lr=lr, max_iter=max_iter)

        def closure():
            optimizer.zero_grad()
            loss = F.cross_entropy(self.forward(logits), labels)
            loss.backward()
            return loss

        optimizer.step(closure)
        return self


def fit_temperature(model, val_loader, device):
    val_logits, val_labels = get_logits_labels(model, val_loader, device)
    scaler = TemperatureScaler().fit(val_logits, val_labels)
    return scaler, val_logits, val_labels


resnet_scaler, resnet_val_logits, resnet_val_labels = fit_temperature(resnet, val_loader, device)
convnet_scaler, convnet_val_logits, convnet_val_labels = fit_temperature(convnet, val_loader, device)

print(f"ResNet-56 learned temperature T   = {resnet_scaler.T.item():.3f}")
print(f"SmallConvNet learned temperature T = {convnet_scaler.T.item():.3f}")


In [ ]:
resnet_test_logits_scaled = resnet_scaler(resnet_test_logits).detach()
resnet_conf_ts, resnet_correct_ts, resnet_pred_ts = confidence_and_correctness(resnet_test_logits_scaled, test_labels)
_, _, resnet_pred_orig = confidence_and_correctness(resnet_test_logits, test_labels)

print(f"Accuracy unchanged by temperature scaling: {torch.equal(resnet_pred_ts, resnet_pred_orig)}")
print(f"ResNet-56 uncalibrated : ECE={compute_ece(resnet_conf, resnet_correct)*100:.2f}%  "
      f"NLL={F.cross_entropy(resnet_test_logits, test_labels).item():.4f}")
print(f"ResNet-56 temp. scaled : ECE={compute_ece(resnet_conf_ts, resnet_correct_ts)*100:.2f}%  "
      f"NLL={F.cross_entropy(resnet_test_logits_scaled, test_labels).item():.4f}")


## 10. Histogram binning (Eq. 7)

The paper's histogram binning is a one-vs-rest scheme over all `K` classes. To keep this small-scale notebook
focused, we implement the common **simplified top-label version**: bin the validation set's *max-softmax
confidence* into `M` equal-width bins, and set each bin's calibrated output to the bin's empirical accuracy on
the validation set (this is exactly Eq. 7's solution, `theta_m` = fraction of correct predictions in bin `m`,
applied directly to the top-label confidence rather than to all `K` one-vs-rest problems).


In [ ]:
def histogram_binning_fit(confidences, correct, n_bins=15):
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    thetas = np.zeros(n_bins)
    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i + 1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        thetas[i] = correct[mask].mean() if mask.sum() > 0 else (lo + hi) / 2
    return bin_boundaries, thetas


def histogram_binning_apply(confidences, bin_boundaries, thetas):
    n_bins = len(thetas)
    out = np.zeros_like(confidences)
    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i + 1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        out[mask] = thetas[i]
    return out


resnet_val_conf, resnet_val_correct, _ = confidence_and_correctness(resnet_val_logits, resnet_val_labels)
bin_boundaries, thetas = histogram_binning_fit(resnet_val_conf, resnet_val_correct, n_bins=15)
resnet_conf_hb = histogram_binning_apply(resnet_conf, bin_boundaries, thetas)

print(f"ResNet-56 hist. binning: ECE={compute_ece(resnet_conf_hb, resnet_correct)*100:.2f}%")


## 11. Results: Table 1 (subset) and Figure 4 style diagrams

A small-scale version of the paper's Table 1 (ECE before/after calibration) and Figure 4 (reliability diagrams
before/after calibration), for our two CIFAR-100 models.


In [ ]:
convnet_conf_hb = histogram_binning_apply(
    convnet_conf, *histogram_binning_fit(*confidence_and_correctness(convnet_val_logits, convnet_val_labels)[:2], n_bins=15)
)
convnet_test_logits_scaled = convnet_scaler(convnet_test_logits).detach()
convnet_conf_ts, convnet_correct_ts, _ = confidence_and_correctness(convnet_test_logits_scaled, test_labels)

results = pd.DataFrame([
    {
        "Model": "SmallConvNet",
        "Accuracy": convnet_correct.mean(),
        "ECE Uncalibrated": compute_ece(convnet_conf, convnet_correct),
        "ECE Temp. Scaling": compute_ece(convnet_conf_ts, convnet_correct_ts),
        "ECE Hist. Binning": compute_ece(convnet_conf_hb, convnet_correct),
    },
    {
        "Model": "ResNet-56",
        "Accuracy": resnet_correct.mean(),
        "ECE Uncalibrated": compute_ece(resnet_conf, resnet_correct),
        "ECE Temp. Scaling": compute_ece(resnet_conf_ts, resnet_correct_ts),
        "ECE Hist. Binning": compute_ece(resnet_conf_hb, resnet_correct),
    },
])
for col in results.columns:
    if col.startswith("ECE"):
        results[col] = (results[col] * 100).round(2)
results["Accuracy"] = (results["Accuracy"] * 100).round(2)
results


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
reliability_diagram(axes[0], resnet_conf, resnet_correct, title="Uncalibrated — ResNet-56")
reliability_diagram(axes[1], resnet_conf_ts, resnet_correct_ts, title="Temp. Scaling — ResNet-56")
reliability_diagram(axes[2], resnet_conf_hb, resnet_correct, title="Hist. Binning — ResNet-56")
plt.tight_layout()
plt.show()


## 12. Discussion

Reproduced (qualitatively, at small scale):
- **Deeper/higher-capacity networks are more miscalibrated**, even though they're more accurate — the ResNet's
  confidence histogram should skew towards 1.0 much more than the ConvNet's, and its reliability-diagram gap
  (red) should be larger.
- **Temperature scaling is remarkably effective for how simple it is** — a single scalar fit on a validation
  set substantially reduces ECE for the ResNet, and leaves accuracy exactly unchanged (verified above by
  comparing arg-max predictions before/after).
- **Histogram binning** also reduces ECE, but — as the paper notes — is noisier bin-to-bin than temperature
  scaling, especially with a small validation set like ours (5,000 samples / 15 bins ≈ 330 samples/bin on average,
  unevenly distributed).

What's *not* expected to match the paper closely: absolute accuracy and ECE numbers (fewer epochs, a shallower
ResNet, and a single random seed will all shift results relative to Table 1's fully-trained, multi-architecture
sweep).


## Appendix (optional): effect of depth on calibration (Figure 2, mini version)

The paper's Figure 2 (far left) shows ECE increasing with ResNet depth on CIFAR-100, even as error stays roughly
flat or drops. This cell reproduces that *trend* by training a few ResNets of increasing depth for a reduced
number of epochs. **This is compute-heavy (trains 3–4 more networks) — skip it unless you have time/GPU budget
to spare.**


In [ ]:
RUN_DEPTH_APPENDIX = False  # flip to True to run this section

if RUN_DEPTH_APPENDIX:
    depths = [8, 20, 32, 56]
    depth_epochs = 20
    depth_results = []
    for d in depths:
        print(f"--- depth {d} ---")
        m = CifarResNet(depth=d, num_classes=NUM_CLASSES)
        fit(m, train_loader, val_loader, device, epochs=depth_epochs, lr=0.1, weight_decay=5e-4,
            milestones=[int(depth_epochs * 0.5), int(depth_epochs * 0.75)])
        logits, labels = get_logits_labels(m, test_loader, device)
        conf, correct, _ = confidence_and_correctness(logits, labels)
        depth_results.append({"depth": d, "error": 1 - correct.mean(), "ece": compute_ece(conf, correct)})

    depth_df = pd.DataFrame(depth_results)
    fig, ax1 = plt.subplots(figsize=(6, 4))
    ax2 = ax1.twinx()
    ax1.plot(depth_df["depth"], depth_df["error"] * 100, "o-", color="tab:blue", label="Error")
    ax2.plot(depth_df["depth"], depth_df["ece"] * 100, "o-", color="tab:red", label="ECE")
    ax1.set_xlabel("Depth"); ax1.set_ylabel("Error (%)", color="tab:blue"); ax2.set_ylabel("ECE (%)", color="tab:red")
    ax1.set_title("Varying Depth — ResNet — CIFAR-100 (mini reproduction of Figure 2)")
    fig.tight_layout()
    plt.show()
    display(depth_df)
